# SciFact and FiQA

Both sets are [BEIR](https://github.com/beir-cellar/beir) collections that RAGBench scores retrieval against. On disk each one has three files:

- `corpus.jsonl` — passages (`_id`, `title`, `text`)
- `queries.jsonl` — questions or claims (`_id`, `text`)
- `qrels/{split}.tsv` — which passages are relevant to which query

RAGBench evaluates the **test** split (`configs/scifact.yaml`, `configs/fiqa.yaml`). The first time FiQA is missing, the next cell downloads it into `data/fiqa`.

In [1]:
import json
from collections import Counter
from pathlib import Path

import pandas as pd
from IPython.display import display

from ragbench.data.loader import load_dataset

ROOT = Path.cwd()
if not (ROOT / "configs").exists() and (ROOT.parent / "configs").exists():
    ROOT = ROOT.parent

def word_count(text: str) -> int:
    return len(text.split())

def profile(name: str, split: str = "test"):
    """Load a BEIR dataset the same way the eval runner does."""
    folder = ROOT / "data" / name
    corpus, queries, qrels = load_dataset(name, split=split, local_path=folder)
    gold_per_query = [len(docs) for docs in qrels.values()]
    passage_words = pd.Series([word_count(doc["text"]) for doc in corpus.values()])
    query_words = pd.Series([word_count(text) for text in queries.values()])
    summary = {
        "dataset": name,
        "split": split,
        "passages": len(corpus),
        "passages_with_title": sum(bool(doc["title"].strip()) for doc in corpus.values()),
        "queries_in_file": sum(1 for line in (folder / "queries.jsonl").open() if line.strip()),
        "judged_queries": len(queries),
        "qrel_pairs": sum(gold_per_query),
        "gold_docs_per_query_median": int(pd.Series(gold_per_query).median()),
        "gold_docs_per_query_max": max(gold_per_query),
        "passage_words_median": int(passage_words.median()),
        "query_words_median": int(query_words.median()),
    }
    return corpus, queries, qrels, summary, passage_words, query_words

def preview(corpus, queries, qrels, n=2, words=55):
    """Print a few queries next to a short slice of a relevant passage."""
    shown = 0
    for qid, text in queries.items():
        print(f"Query {qid}: {text}\n")
        for doc_id, score in list(qrels[qid].items())[:1]:
            doc = corpus[doc_id]
            snippet = " ".join(doc["text"].split()[:words])
            title = doc["title"].strip() or "(no title)"
            print(f"  relevant {doc_id} (score {score}) — {title}")
            print(f"  {snippet} ...\n")
        shown += 1
        if shown >= n:
            break

## SciFact

Scientific **claim verification**. A query is a one-sentence claim. A passage is a paper abstract. A relevant abstract is evidence about that claim.

The qrels used for scoring are binary (`score = 1`). Whether the abstract *supports* or *contradicts* the claim is stored separately, in `metadata` on the query.

In [2]:
scifact_corpus, scifact_queries, scifact_qrels, scifact_summary, scifact_passage_words, scifact_query_words = profile("scifact")
display(pd.Series(scifact_summary, name="scifact"))

labels = Counter()
examples = {}
with (ROOT / "data/scifact/queries.jsonl").open() as fh:
    for line in fh:
        row = json.loads(line)
        if str(row["_id"]) not in scifact_qrels:
            continue
        for doc_id, annotations in (row.get("metadata") or {}).items():
            for ann in annotations:
                label = ann.get("label", "UNKNOWN")
                labels[label] += 1
                examples.setdefault(label, (str(row["_id"]), row["text"], str(doc_id)))

print("Evidence labels on the test claims:", dict(labels))
print("Gold abstracts per test claim:")
display(pd.Series([len(docs) for docs in scifact_qrels.values()]).value_counts().sort_index().rename("queries"))
print("Passage length (words):")
display(scifact_passage_words.describe().round(1))

09:47:16 | INFO    | ragbench.data.loader | Loaded 'scifact' [test]: 5183 passages, 300 queries, 300 qrels


dataset                       scifact
split                            test
passages                         5183
passages_with_title              5183
queries_in_file                  1109
judged_queries                    300
qrel_pairs                        339
gold_docs_per_query_median          1
gold_docs_per_query_max             5
passage_words_median              192
query_words_median                 12
Name: scifact, dtype: object

Evidence labels on the test claims: {'SUPPORT': 216, 'CONTRADICT': 122}
Gold abstracts per test claim:


1    277
2     14
3      4
4      3
5      2
Name: queries, dtype: int64

Passage length (words):


count    5183.0
mean      201.8
std        86.0
min        26.0
25%       147.0
50%       192.0
75%       247.0
max      1524.0
dtype: float64

In [3]:
preview(scifact_corpus, scifact_queries, scifact_qrels, n=1)

print("--- one claim of each evidence type ---\n")
for label, (qid, text, doc_id) in examples.items():
    doc = scifact_corpus[doc_id]
    print(f"{label}  query {qid}: {text}")
    print(f"  abstract {doc_id}: {doc['title']}")
    print(f"  {' '.join(doc['text'].split()[:40])} ...\n")

Query 1: 0-dimensional biomaterials show inductive properties.

  relevant 31715818 (score 1) — New opportunities: the use of nanotechnologies to manipulate and track stem cells.
  Nanotechnologies are emerging platforms that could be useful in measuring, understanding, and manipulating stem cells. Examples include magnetic nanoparticles and quantum dots for stem cell labeling and in vivo tracking; nanoparticles, carbon nanotubes, and polyplexes for the intracellular delivery of genes/oligonucleotides and protein/peptides; and engineered nanometer-scale scaffolds for stem cell differentiation and transplantation. This review ...

--- one claim of each evidence type ---

SUPPORT  query 3: 1,000 genomes project enables mapping of genetic sequence variation consisting of rare variants with larger penetrance effects than common variants.
  abstract 14717500: Rare Variants Create Synthetic Genome-Wide Associations
  Genome-wide association studies (GWAS) have now identified at least 2,000 c

## FiQA

Financial **question answering** (FiQA-2018). A query is a user question, often from an investment forum. A passage is an answer-style post. Relevance means the passage addresses the question. There is no support/contradict label.

This is the larger benchmark RAGBench is built around. Expect the download on the first run.

In [4]:
fiqa_corpus, fiqa_queries, fiqa_qrels, fiqa_summary, fiqa_passage_words, fiqa_query_words = profile("fiqa")
display(pd.Series(fiqa_summary, name="fiqa"))
print("Gold passages per test question:")
display(pd.Series([len(docs) for docs in fiqa_qrels.values()]).value_counts().sort_index().rename("queries").head(8))
print("Passage length (words):")
display(fiqa_passage_words.describe().round(1))
preview(fiqa_corpus, fiqa_queries, fiqa_qrels, n=2)

09:47:16 | INFO    | ragbench.data.loader | Downloading BEIR dataset 'fiqa' from https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/fiqa.zip


09:47:20 | INFO    | ragbench.data.loader | Unzipped 'fiqa' -> /Users/sushantpatil/sushant/ragbench/data/fiqa


09:47:21 | INFO    | ragbench.data.loader | Loaded 'fiqa' [test]: 57638 passages, 648 queries, 648 qrels


dataset                        fiqa
split                          test
passages                      57638
passages_with_title               0
queries_in_file                6648
judged_queries                  648
qrel_pairs                     1706
gold_docs_per_query_median        2
gold_docs_per_query_max          15
passage_words_median             90
query_words_median               10
Name: fiqa, dtype: object

Gold passages per test question:


1    220
2    184
3    103
4     56
5     29
6     20
7     13
8      8
Name: queries, dtype: int64

Passage length (words):


count    57638.0
mean       132.9
std        128.7
min          0.0
25%         57.0
50%         90.0
75%        159.0
max       2973.0
dtype: float64

Query 4641: Where should I park my rainy-day / emergency fund?

  relevant 44594 (score 1) — (no title)
  "First off, you generally want to park your emergency fund somewhere that is ""safe"", meaning something that is not subject to market fluctuations. Your emergency fund is something you need to be able to count on when times are tough! That rules out things like stock market investments. Secondly, you need to think about how ...

Query 5503: Tax considerations for selling a property below appraised value to family?

  relevant 146277 (score 1) — (no title)
  "Is this legal? Why not? But you might have trouble deducting losses on your taxes, especially if you sell to someone related to you in some way (which is indeed what you're doing). See the added portion below regarding dealing with ""related person"" (which a sibling is). The state of Maryland has a transfer/recordation tax ...



## Side by side

SciFact is a small, titled, abstract-length corpus of scientific claims, usually with a single gold document. FiQA is a large corpus of untitled forum answers, and a question often has several relevant posts. SciFact is the set to iterate on. FiQA is the set the benchmark is meant to measure.

In [5]:
comparison = pd.DataFrame([scifact_summary, fiqa_summary]).set_index("dataset")
display(comparison.T)

dataset,scifact,fiqa
split,test,test
passages,5183,57638
passages_with_title,5183,0
queries_in_file,1109,6648
judged_queries,300,648
qrel_pairs,339,1706
gold_docs_per_query_median,1,2
gold_docs_per_query_max,5,15
passage_words_median,192,90
query_words_median,12,10
